# E8 — Same-manufacturer exclusion sensitivity analysis
## Does the E6 DynamicReal advantage persist when same-manufacturer batteries are removed from training?

This notebook is a targeted robustness follow-up to E6. It reuses the **E6 v2**
architecture, post-current dynamic-state convention, deferred node-deactivation rule,
training objective, hyperparameters, three neural seeds, and tie-aware ranking metrics.

### Why E8 is needed

Strict leave-one-product-out (LOPO) evaluation holds out one battery product, but for
BMW and Chevrolet the training set can still contain another battery from the same
manufacturer. E8 asks whether the key E6 mechanism result survives after removing
**all same-manufacturer products** from the corresponding training fold.

Only the five affected held-out products are evaluated:

- BMW 530e
- BMW 330e
- BMW i3
- Chevrolet Spark
- Chevrolet Volt

For a held-out BMW, the other two BMW batteries are excluded from training.
For a held-out Chevrolet, the other Chevrolet battery is excluded from training.
All products from other manufacturers remain in training.

### Conditions

E8 reruns only the mechanism-critical pair:

1. **DynamicRealGNN**
2. **DynamicShuffledGNN × 5**

The dynamic shuffled controls use the same fixed E6 permutation rule for the initial
graph and every later structural event, preserving the evolving unlabeled graph
sequence up to node relabeling while breaking semantic--topological alignment.

### Statistical scope

The independent unit remains the held-out battery product. E8 has only **five**
affected products, so it is a **sensitivity analysis**, not a new confirmatory study.
The designated comparison is DynamicReal minus the five-shuffle mean on Hits@1.
With n=5, even a 5/5 same-direction result has a minimum possible two-sided exact
sign-flip p-value of 0.0625. Effect size and directional consistency are therefore
the main quantities of interest. Bootstrap intervals are descriptive.

E8 does not alter or replace the n=10 E6 inference.


In [ ]:
# 1. Imports and frozen E6-v2 configuration

from pathlib import Path
from collections import Counter, defaultdict
import tarfile, json, math, random, time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

SEEDS = [17, 29, 43]
SHUFFLE_IDS = [1, 2, 3, 4, 5]

EPOCHS = 30
NEGATIVES_PER_STATE = 8
LR = 2e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10

EMBED_DIM = 12
HIDDEN_DIM = 64
GNN_DIM = 32

BASE_SEED = 20261003
GRAPH_SHUFFLE_BASE = 20261119
REWIRE_BASE = 20270117
REWIRE_IDS = [1, 2, 3, 4, 5]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

E8_EXPECTED_PRODUCTS = 5
E8_CONDITIONS_PER_SEED = 1 + len(SHUFFLE_IDS)
E8_EXPECTED_FITS = (
    E8_EXPECTED_PRODUCTS
    * len(SEEDS)
    * E8_CONDITIONS_PER_SEED
)

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Planned E8 neural fits:", E8_EXPECTED_FITS)

if DEVICE.type != "cuda":
    print("WARNING: GPU is strongly recommended for this notebook.")


In [ ]:
# 2. Find the KIT dataset automatically.
#
# The cell first searches for already-extracted CSVs anywhere under /content.
# If not found, it searches for the uploaded KIT .tar archive and extracts it.

root = Path("/content")

catalogues = list(root.rglob("battery_catalogue.csv"))

if catalogues:
    DATA = catalogues[0].parent
else:
    archives = list(root.rglob("10.35097-emz24pksshndq468.tar"))
    if not archives:
        archives = list(root.rglob("10.35097-*.tar"))

    assert archives, (
        "KIT dataset not found. Upload 10.35097-emz24pksshndq468.tar "
        "to the Colab Files panel, then rerun this cell."
    )

    TAR_PATH = archives[0]
    print("Using archive:", TAR_PATH)

    work = root / "battery_disassembly_dataset_extracted"
    work.mkdir(parents=True, exist_ok=True)

    with tarfile.open(TAR_PATH, "r") as tf:
        tf.extractall(work)

    catalogues = list(work.rglob("battery_catalogue.csv"))
    assert catalogues, "battery_catalogue.csv not found after extraction."
    DATA = catalogues[0].parent

print("Dataset directory:", DATA)

In [ ]:
# 3. Load and validate raw data.

bat = pd.read_csv(DATA / "battery_catalogue.csv")
parts = pd.read_csv(DATA / "parts.csv")
fix = pd.read_csv(DATA / "fixations.csv")
ops = pd.read_csv(DATA / "operations.csv")

print("Batteries:", len(bat))
print("Parts:", len(parts))
print("Fixations:", len(fix))
print("Operations:", len(ops))

display(bat[["id", "name"]])

In [ ]:
# 4. Attach each operation to a battery and reconstruct the 10 reference chains.

battery_names = bat.set_index("id")["name"].to_dict()

fix_battery = fix.set_index("id")["battery_id"].to_dict()
part_battery = parts.set_index("id")["battery_id"].to_dict()
part_class_raw = parts.set_index("id")["class"].to_dict()

fix_info_raw = (
    fix.set_index("id")[["class", "fromPartID", "toPartID"]]
    .to_dict("index")
)

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])

ops["battery_id"] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index("id")

chains = {}

for b in sorted(ops["battery_id"].unique()):
    sub = ops[ops["battery_id"] == b]
    ids = set(sub["id"].astype(int))
    next_ids = set(sub["next_op_id"].dropna().astype(int))

    starts = list(ids - next_ids)
    terminals = sub[sub["next_op_id"].isna()]["id"].astype(int).tolist()

    assert len(starts) == 1
    assert len(terminals) == 1

    seq = []
    seen = set()
    cur = int(starts[0])

    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)

        nxt = ops_by_id.loc[cur, "next_op_id"]
        if pd.isna(nxt):
            break
        cur = int(nxt)

    assert len(seq) == len(ids), f"Broken chain for battery {b}"
    chains[int(b)] = seq

print({
    battery_names[b]: len(seq)
    for b, seq in chains.items()
})

In [ ]:
# 5. Raw full relational signatures.
#
# These are used only for diagnostics (seen vs unseen motif/signature).
# They are NOT output classes for the neural ranking models.

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)

    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi["fromPartID"])])
        dst_cls = str(part_class_raw[int(fi["toPartID"])])
        return (
            f"{skill}|fix:{fi['class']}|"
            f"{src_cls}->{dst_cls}"
        )

    return (
        f"{skill}|part:"
        f"{part_class_raw[int(r.part_id)]}"
    )

raw_signatures = {
    b: [raw_full_signature(op_id) for op_id in seq]
    for b, seq in chains.items()
}

In [ ]:
# 6. Fold-specific vocabularies.
#
# Vocabularies are learned ONLY from the 9 training batteries.
# Category 0 is <UNK>, so truly unseen test categories do not leak into training.

def make_vocab(values):
    vals = sorted({
        str(x)
        for x in values
        if pd.notna(x)
    })
    return {
        "<UNK>": 0,
        **{v: i + 1 for i, v in enumerate(vals)}
    }

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {
        "part": make_vocab(
            parts[
                parts["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "fix": make_vocab(
            fix[
                fix["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "skill": make_vocab(
            ops[
                ops["battery_id"].isin(train_batteries)
            ]["skill_id"]
        ),
    }

In [ ]:
# 7. Convert one battery into product-invariant graph + operation features.
#
# graph_mode:
#   "real"    -> true physical fixation topology
#   "shuffle" -> fixed permutation of topology relative to semantic/operation nodes
#   "noedge"  -> same GNN architecture, but no message-passing edges
#
# Operation-local attributes are identical across all graph modes.
# Therefore only the graph branch is manipulated.

def build_battery_pack(
    battery_id,
    vocab,
    graph_mode="real",
    shuffle_id=0,
):
    assert graph_mode in {"real", "shuffle", "noedge"}

    psub = parts[
        parts["battery_id"] == battery_id
    ].copy()

    fsub = fix[
        fix["battery_id"] == battery_id
    ].copy()

    part_id_to_local = {
        int(pid): i
        for i, pid in enumerate(psub["id"].astype(int))
    }

    n_nodes = len(psub)

    true_degree = np.zeros(n_nodes, dtype=np.float32)

    edge_src = []
    edge_dst = []
    edge_type = []

    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]

        true_degree[a] += 1
        true_degree[b] += 1

        # Undirected message-passing representation.
        edge_src += [a, b]
        edge_dst += [b, a]

        et = lookup(vocab["fix"], r["class"])
        edge_type += [et, et]

    true_degree_norm = (
        true_degree / max(1.0, float(true_degree.max()))
    )

    node_part_class = np.array(
        [
            lookup(vocab["part"], cls)
            for cls in psub["class"]
        ],
        dtype=np.int64,
    )

    true_edge_index = np.array(
        [edge_src, edge_dst],
        dtype=np.int64,
    )

    true_edge_type = np.array(
        edge_type,
        dtype=np.int64,
    )

    if graph_mode == "real":
        graph_edge_index = true_edge_index.copy()
        graph_edge_type = true_edge_type.copy()
        graph_degree_norm = true_degree_norm.copy()

    elif graph_mode == "shuffle":
        # A deterministic, independent permutation for each
        # (shuffle_id, battery). This preserves the unlabeled graph,
        # but reassigns its physical locations to semantic nodes.
        rng_graph = np.random.default_rng(
            GRAPH_SHUFFLE_BASE
            + 10007 * int(shuffle_id)
            + int(battery_id)
        )

        permutation = rng_graph.permutation(n_nodes)

        graph_edge_index = permutation[true_edge_index]
        graph_edge_type = true_edge_type.copy()

        shuffled_degree = np.bincount(
            graph_edge_index.reshape(-1),
            minlength=n_nodes,
        ).astype(np.float32)

        graph_degree_norm = (
            shuffled_degree
            / max(1.0, float(shuffled_degree.max()))
        )

    else:  # noedge
        graph_edge_index = np.empty((2, 0), dtype=np.int64)
        graph_edge_type = np.empty((0,), dtype=np.int64)

        # Do not leak adjacency through the graph branch.
        graph_degree_norm = np.zeros(
            n_nodes,
            dtype=np.float32,
        )

    seq = [int(x) for x in chains[battery_id]]

    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []

    for op_id in seq:
        r = ops_by_id.loc[op_id]

        skill_idx = lookup(
            vocab["skill"],
            r.skill_id,
        )

        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]

            src_pid = int(fi["fromPartID"])
            dst_pid = int(fi["toPartID"])

            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]

            categorical.append([
                skill_idx,
                0,
                lookup(vocab["fix"], fi["class"]),
                lookup(vocab["part"], part_class_raw[src_pid]),
                lookup(vocab["part"], part_class_raw[dst_pid]),
                0,
            ])

            graph_nodes.append([
                src_node,
                dst_node,
                -1,
            ])

            # IMPORTANT:
            # identical operation-local features in every graph condition.
            numeric.append([
                true_degree_norm[src_node],
                true_degree_norm[dst_node],
                0.0,
            ])

        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]

            categorical.append([
                skill_idx,
                1,
                0,
                0,
                0,
                lookup(vocab["part"], part_class_raw[pid]),
            ])

            graph_nodes.append([
                -1,
                -1,
                node,
            ])

            numeric.append([
                0.0,
                0.0,
                true_degree_norm[node],
            ])

        signatures.append(raw_full_signature(op_id))

    return {
        "battery_id": int(battery_id),
        "battery_name": battery_names[int(battery_id)],
        "node_part_class": node_part_class,
        "node_degree": graph_degree_norm.astype(np.float32),
        "edge_index": graph_edge_index,
        "edge_type": graph_edge_type,
        "op_categorical": np.asarray(categorical, dtype=np.int64),
        "op_graph_nodes": np.asarray(graph_nodes, dtype=np.int64),
        "op_numeric": np.asarray(numeric, dtype=np.float32),
        "full_signatures": signatures,
        "n_ops": len(seq),
        "graph_mode": graph_mode,
        "shuffle_id": int(shuffle_id),
    }

In [ ]:
# 8. Neural models.

class BaseOperationRanker(nn.Module):
    """
    Product-invariant MLP operation ranker.

    It never receives battery/product IDs or raw entity IDs.
    """

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.skill_emb = nn.Embedding(n_skill, emb_dim)
        self.kind_emb = nn.Embedding(2, 4)
        self.fix_emb = nn.Embedding(n_fix, emb_dim)
        self.part_emb = nn.Embedding(n_part, emb_dim)

        op_input_dim = (
            emb_dim      # skill
            + 4          # operation kind
            + emb_dim    # fixation class
            + emb_dim    # source part class
            + emb_dim    # target part class
            + emb_dim    # removed part class
            + 3          # normalized local degrees
        )

        self.op_mlp = nn.Sequential(
            nn.Linear(op_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # current, candidate, abs difference, product, progress
        self.score_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def base_operation_encoding(self, pack):
        cats = torch.as_tensor(
            pack["op_categorical"],
            dtype=torch.long,
            device=DEVICE,
        )

        nums = torch.as_tensor(
            pack["op_numeric"],
            dtype=torch.float32,
            device=DEVICE,
        )

        skill = cats[:, 0]
        kind = cats[:, 1]
        fix_class = cats[:, 2]
        src_class = cats[:, 3]
        dst_class = cats[:, 4]
        part_class = cats[:, 5]

        x = torch.cat([
            self.skill_emb(skill),
            self.kind_emb(kind),
            self.fix_emb(fix_class),
            self.part_emb(src_class),
            self.part_emb(dst_class),
            self.part_emb(part_class),
            nums,
        ], dim=-1)

        return self.op_mlp(x)

    def encode_operations(self, pack):
        return self.base_operation_encoding(pack)

    def score_pairs(self, current_repr, candidate_repr, progress):
        x = torch.cat([
            current_repr,
            candidate_repr,
            torch.abs(current_repr - candidate_repr),
            current_repr * candidate_repr,
            progress,
        ], dim=-1)

        return self.score_mlp(x).squeeze(-1)


class GraphEncoder(nn.Module):
    """
    Two-layer product-invariant message-passing network.
    """

    def __init__(
        self,
        n_part,
        n_fix,
        part_emb_dim=EMBED_DIM,
        edge_emb_dim=8,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.part_emb = nn.Embedding(n_part, part_emb_dim)
        self.edge_emb = nn.Embedding(n_fix, edge_emb_dim)

        self.input_proj = nn.Linear(
            part_emb_dim + 1,
            graph_dim,
        )

        self.message_layers = nn.ModuleList([
            nn.Linear(
                graph_dim + edge_emb_dim,
                graph_dim,
            )
            for _ in range(2)
        ])

        self.self_layers = nn.ModuleList([
            nn.Linear(graph_dim, graph_dim)
            for _ in range(2)
        ])

        self.dropout = dropout

    def forward(self, pack):
        node_class = torch.as_tensor(
            pack["node_part_class"],
            dtype=torch.long,
            device=DEVICE,
        )

        degree = torch.as_tensor(
            pack["node_degree"],
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(-1)

        h = F.relu(
            self.input_proj(
                torch.cat([
                    self.part_emb(node_class),
                    degree,
                ], dim=-1)
            )
        )

        edge_index = torch.as_tensor(
            pack["edge_index"],
            dtype=torch.long,
            device=DEVICE,
        )

        edge_type = torch.as_tensor(
            pack["edge_type"],
            dtype=torch.long,
            device=DEVICE,
        )

        src = edge_index[0]
        dst = edge_index[1]

        edge_embedding = self.edge_emb(edge_type)

        for message_layer, self_layer in zip(
            self.message_layers,
            self.self_layers,
        ):
            msg = message_layer(
                torch.cat([
                    h[src],
                    edge_embedding,
                ], dim=-1)
            )

            agg = torch.zeros(
                (h.shape[0], h.shape[1]),
                dtype=h.dtype,
                device=DEVICE,
            )

            agg.index_add_(0, dst, msg)

            counts = torch.zeros(
                (h.shape[0], 1),
                dtype=h.dtype,
                device=DEVICE,
            )

            counts.index_add_(
                0,
                dst,
                torch.ones(
                    (dst.numel(), 1),
                    dtype=h.dtype,
                    device=DEVICE,
                ),
            )

            agg = agg / counts.clamp_min(1.0)

            h = F.relu(
                self_layer(h) + agg
            )

            h = F.dropout(
                h,
                p=self.dropout,
                training=self.training,
            )

        return h


class GNNOperationRanker(BaseOperationRanker):

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__(
            n_skill=n_skill,
            n_fix=n_fix,
            n_part=n_part,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.graph_encoder = GraphEncoder(
            n_part=n_part,
            n_fix=n_fix,
            part_emb_dim=emb_dim,
            graph_dim=graph_dim,
            dropout=dropout,
        )

        # src embedding, dst embedding, absolute difference
        self.structural_projection = nn.Sequential(
            nn.Linear(graph_dim * 3, hidden_dim),
            nn.ReLU(),
        )

        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
        )

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)
        node_h = self.graph_encoder(pack)

        op_nodes = torch.as_tensor(
            pack["op_graph_nodes"],
            dtype=torch.long,
            device=DEVICE,
        )

        n_ops = op_nodes.shape[0]

        a = torch.zeros(
            (n_ops, node_h.shape[1]),
            device=DEVICE,
        )

        b = torch.zeros_like(a)

        fixation_mask = op_nodes[:, 0] >= 0

        if fixation_mask.any():
            a[fixation_mask] = node_h[
                op_nodes[fixation_mask, 0]
            ]

            b[fixation_mask] = node_h[
                op_nodes[fixation_mask, 1]
            ]

        part_mask = op_nodes[:, 2] >= 0

        if part_mask.any():
            a[part_mask] = node_h[
                op_nodes[part_mask, 2]
            ]

        structural = self.structural_projection(
            torch.cat([
                a,
                b,
                torch.abs(a - b),
            ], dim=-1)
        )

        return self.combine(
            torch.cat([
                base,
                structural,
            ], dim=-1)
        )

In [ ]:
# 9. Reproducibility + pairwise-ranking training

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_training_pairs(n_ops, k, rng):
    current, positive, negative, progress = [], [], [], []

    for t in range(n_ops - 2):
        pool = np.arange(
            t + 2,
            n_ops,
            dtype=np.int64,
        )

        neg = rng.choice(
            pool,
            size=k,
            replace=(len(pool) < k),
        )

        current.append(t)
        positive.append(t + 1)
        negative.append(neg)
        progress.append(
            t / max(1, n_ops - 1)
        )

    return (
        np.asarray(current, dtype=np.int64),
        np.asarray(positive, dtype=np.int64),
        np.stack(negative),
        np.asarray(progress, dtype=np.float32),
    )


def build_gnn(vocab):
    model = GNNOperationRanker(
        n_skill=len(vocab["skill"]),
        n_fix=len(vocab["fix"]),
        n_part=len(vocab["part"]),
    )
    return model.to(DEVICE)


def train_one_gnn_condition(
    train_batteries,
    test_battery,
    seed,
    graph_mode,
    shuffle_id=0,
):
    set_seed(seed)

    vocab = build_fold_vocab(train_batteries)

    packs = {
        b: build_battery_pack(
            b,
            vocab,
            graph_mode=graph_mode,
            shuffle_id=shuffle_id,
        )
        for b in train_batteries + [test_battery]
    }

    model = build_gnn(vocab)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    # Using the same training seed across paired graph conditions
    # keeps initialization, training order, and negative sampling paired.
    rng = np.random.default_rng(seed)

    for epoch in range(EPOCHS):
        order = list(train_batteries)
        rng.shuffle(order)

        model.train()

        for b in order:
            pack = packs[b]

            cur, pos, neg, prog = make_training_pairs(
                pack["n_ops"],
                NEGATIVES_PER_STATE,
                rng,
            )

            z = model.encode_operations(pack)

            cur_t = torch.as_tensor(
                cur,
                dtype=torch.long,
                device=DEVICE,
            )

            pos_t = torch.as_tensor(
                pos,
                dtype=torch.long,
                device=DEVICE,
            )

            neg_t = torch.as_tensor(
                neg,
                dtype=torch.long,
                device=DEVICE,
            )

            prog_t = torch.as_tensor(
                prog,
                dtype=torch.float32,
                device=DEVICE,
            ).unsqueeze(-1)

            cur_z = z[cur_t]
            pos_z = z[pos_t]

            pos_score = model.score_pairs(
                cur_z,
                pos_z,
                prog_t,
            )

            k = neg_t.shape[1]

            cur_rep = (
                cur_z[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, cur_z.shape[1])
            )

            neg_z = z[neg_t.reshape(-1)]

            prog_rep = (
                prog_t[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, 1)
            )

            neg_score = model.score_pairs(
                cur_rep,
                neg_z,
                prog_rep,
            ).view(len(cur), k)

            loss = F.softplus(
                -(pos_score[:, None] - neg_score)
            ).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model, packs[test_battery], vocab

In [ ]:
# 10. Tie-aware ranking metrics.

def expected_tie_metrics(scores, target_position=0, tol=1e-8):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """

    scores = np.asarray(scores)
    target_score = scores[target_position]

    greater = int(
        np.sum(scores > target_score + tol)
    )

    equal = int(
        np.sum(
            np.abs(scores - target_score) <= tol
        )
    )

    equal = max(equal, 1)

    hits1 = (
        max(
            0,
            min(equal, 1 - greater),
        )
        / equal
        if greater < 1
        else 0.0
    )

    hits3 = (
        max(
            0,
            min(equal, 3 - greater),
        )
        / equal
        if greater < 3
        else 0.0
    )

    ranks = np.arange(
        greater + 1,
        greater + equal + 1,
        dtype=float,
    )

    mrr = float(
        np.mean(1.0 / ranks)
    )

    return hits1, hits3, mrr

In [ ]:
# 11. Evaluate one neural model on an unseen battery.

def training_signature_set(train_batteries):
    s = set()

    for b in train_batteries:
        s.update(raw_signatures[b])

    return s


def evaluate_neural_model(
    model,
    test_pack,
    train_batteries,
):
    model.eval()

    seen_signatures = training_signature_set(
        train_batteries
    )

    rows = []

    with torch.no_grad():
        operation_repr = model.encode_operations(
            test_pack
        )

        n_ops = test_pack["n_ops"]

        for t in range(n_ops - 1):
            candidate_indices = np.arange(
                t + 1,
                n_ops,
                dtype=np.int64,
            )

            candidate_t = torch.as_tensor(
                candidate_indices,
                dtype=torch.long,
                device=DEVICE,
            )

            current_repr = (
                operation_repr[t]
                .unsqueeze(0)
                .expand(
                    len(candidate_indices),
                    -1,
                )
            )

            candidate_repr = operation_repr[
                candidate_t
            ]

            progress = torch.full(
                (len(candidate_indices), 1),
                t / max(1, n_ops - 1),
                dtype=torch.float32,
                device=DEVICE,
            )

            scores = model.score_pairs(
                current_repr,
                candidate_repr,
                progress,
            ).cpu().numpy()

            # candidate_indices[0] is the reference next operation.
            h1, h3, mrr = expected_tie_metrics(
                scores,
                target_position=0,
            )

            target_signature = (
                test_pack["full_signatures"][t + 1]
            )

            rows.append({
                "position": t,
                "hits1": h1,
                "hits3": h3,
                "mrr": mrr,
                "signature_seen_in_training":
                    target_signature in seen_signatures,
            })

    detail = pd.DataFrame(rows)

    metrics = {
        "hits1": float(detail["hits1"].mean()),
        "hits3": float(detail["hits3"].mean()),
        "mrr": float(detail["mrr"].mean()),
        "seen_signature_rate": float(
            detail["signature_seen_in_training"].mean()
        ),
    }

    seen = detail[
        detail["signature_seen_in_training"]
    ]

    unseen = detail[
        ~detail["signature_seen_in_training"]
    ]

    for label, subset in [
        ("seen", seen),
        ("unseen", unseen),
    ]:
        if len(subset) > 0:
            metrics[f"{label}_hits1"] = float(
                subset["hits1"].mean()
            )

            metrics[f"{label}_hits3"] = float(
                subset["hits3"].mean()
            )

            metrics[f"{label}_mrr"] = float(
                subset["mrr"].mean()
            )
        else:
            metrics[f"{label}_hits1"] = np.nan
            metrics[f"{label}_hits3"] = np.nan
            metrics[f"{label}_mrr"] = np.nan

    return metrics, detail

In [ ]:
# 12. Exact-capacity validity helper

def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

## 13. Build the state-aware block-diagonal graph sequence

The original `build_battery_pack` supplies invariant operation attributes and operation-to-node references. E6 then constructs one graph block per ranking state.

The revised state update uses **deferred part-node deactivation**. A part operation marks its node as removal-complete, but the node stays structurally active while any modeled fixation incident to it remains unexecuted. Once its last remaining fixation disappears, the node is automatically deactivated. This handles the small ordering irregularity in the Fiat 500e chain without deleting future reference operations and without looking ahead to the next target.

For `dynamic_real`, structural updates are applied to their true semantic nodes. For `dynamic_shuffle`, the initial graph and every later structural event are transformed through the same fixed permutation. For `dynamic_complete`, the same causally derived active-node set is retained but the physical fixation edges are replaced by all-to-all edges with dummy type `0`.


In [ ]:
# 13. Dynamic state-graph construction

from collections import defaultdict

DYNAMIC_SHUFFLE_BASE = GRAPH_SHUFFLE_BASE


def _battery_structure_records(battery_id, vocab):
    psub = parts[
        parts["battery_id"] == battery_id
    ].copy()

    fsub = fix[
        fix["battery_id"] == battery_id
    ].copy()

    part_ids = psub["id"].astype(int).tolist()
    part_id_to_local = {
        pid: i
        for i, pid in enumerate(part_ids)
    }

    records = []

    for _, r in fsub.iterrows():
        records.append({
            "fixation_id": int(r["id"]),
            "u": part_id_to_local[int(r.fromPartID)],
            "v": part_id_to_local[int(r.toPartID)],
            "edge_type": lookup(vocab["fix"], r["class"]),
        })

    return (
        psub,
        part_id_to_local,
        records,
    )


def _dynamic_permutation(
    battery_id,
    n_nodes,
    shuffle_id,
):
    rng = np.random.default_rng(
        DYNAMIC_SHUFFLE_BASE
        + 10007 * int(shuffle_id)
        + int(battery_id)
    )

    return rng.permutation(
        n_nodes
    )


def _state_edges_and_degree(
    n_nodes,
    records,
    removed_fixations,
    active_nodes,
    initial_max_degree,
):
    edge_src = []
    edge_dst = []
    edge_type = []

    degree = np.zeros(
        n_nodes,
        dtype=np.float32,
    )

    for rec in records:
        if rec["fixation_id"] in removed_fixations:
            continue

        u = int(rec["u"])
        v = int(rec["v"])

        if not active_nodes[u] or not active_nodes[v]:
            continue

        degree[u] += 1.0
        degree[v] += 1.0

        edge_src += [u, v]
        edge_dst += [v, u]
        edge_type += [
            int(rec["edge_type"]),
            int(rec["edge_type"]),
        ]

    if len(edge_src):
        edge_index = np.asarray(
            [edge_src, edge_dst],
            dtype=np.int64,
        )
        edge_type = np.asarray(
            edge_type,
            dtype=np.int64,
        )
    else:
        edge_index = np.empty(
            (2, 0),
            dtype=np.int64,
        )
        edge_type = np.empty(
            (0,),
            dtype=np.int64,
        )

    degree_norm = (
        degree
        / max(
            1.0,
            float(initial_max_degree),
        )
    ).astype(np.float32)

    return (
        edge_index,
        edge_type,
        degree_norm,
    )


def _complete_state_edges_and_degree(
    active_nodes,
):
    active = np.flatnonzero(
        active_nodes
    ).astype(np.int64)

    n_nodes = len(
        active_nodes
    )

    if len(active) <= 1:
        return (
            np.empty(
                (2, 0),
                dtype=np.int64,
            ),
            np.empty(
                (0,),
                dtype=np.int64,
            ),
            np.zeros(
                n_nodes,
                dtype=np.float32,
            ),
        )

    src = np.repeat(
        active,
        len(active),
    )
    dst = np.tile(
        active,
        len(active),
    )

    keep = (
        src != dst
    )

    src = src[keep]
    dst = dst[keep]

    edge_index = np.asarray(
        [src, dst],
        dtype=np.int64,
    )

    # Edge-type 0 is the reserved <UNK>/dummy type.
    edge_type = np.zeros(
        len(src),
        dtype=np.int64,
    )

    degree_norm = np.zeros(
        n_nodes,
        dtype=np.float32,
    )

    # Complete-graph normalized degree among the currently active nodes.
    degree_norm[active] = (
        (len(active) - 1)
        / max(
            1,
            n_nodes - 1,
        )
    )

    return (
        edge_index,
        edge_type,
        degree_norm,
    )


def _has_remaining_incident_fixation(
    node,
    records,
    removed_fixations,
    active_nodes,
):
    """
    Causal check: does this currently active node still participate in any
    modeled fixation that has not yet been executed and whose other endpoint
    is also structurally active?
    """
    for rec in records:
        if rec["fixation_id"] in removed_fixations:
            continue

        u = int(rec["u"])
        v = int(rec["v"])

        if not active_nodes[u] or not active_nodes[v]:
            continue

        if node == u or node == v:
            return True

    return False


def _deactivate_completed_parts(
    records,
    removed_fixations,
    active_nodes,
    part_completed,
):
    """
    Deactivate a part only after:
      1) its part operation has already occurred, and
      2) it has no remaining active incident fixation.

    Iterate to a fixed point because deactivating one completed isolated node
    can make another completed node isolated as well.
    """
    changed = True

    while changed:
        changed = False

        pending = np.flatnonzero(
            part_completed
            & active_nodes
        )

        for node in pending:
            node = int(node)

            if not _has_remaining_incident_fixation(
                node=node,
                records=records,
                removed_fixations=removed_fixations,
                active_nodes=active_nodes,
            ):
                active_nodes[node] = False
                changed = True

    return active_nodes


def build_e6_state_pack(
    battery_id,
    vocab,
    condition,
    shuffle_id=0,
):
    """
    Conditions:
      static_real
      dynamic_real
      dynamic_noedge
      dynamic_complete
      dynamic_shuffle

    Returns the invariant operation pack plus one block-diagonal graph
    containing all post-current ranking states t=0,...,n_ops-2.
    """
    assert condition in {
        "static_real",
        "dynamic_real",
        "dynamic_noedge",
        "dynamic_complete",
        "dynamic_shuffle",
    }

    # Invariant operation features are always taken from the canonical
    # real-topology pack. Thus operation-local true degrees are identical
    # across every E6 graph condition.
    base_pack = build_battery_pack(
        battery_id,
        vocab,
        graph_mode="real",
        shuffle_id=0,
    )

    (
        psub,
        part_id_to_local,
        true_records,
    ) = _battery_structure_records(
        battery_id,
        vocab,
    )

    n_nodes = len(psub)
    n_states = base_pack["n_ops"] - 1

    initial_degree = np.zeros(
        n_nodes,
        dtype=np.float32,
    )

    for rec in true_records:
        initial_degree[rec["u"]] += 1.0
        initial_degree[rec["v"]] += 1.0

    initial_max_degree = max(
        1.0,
        float(initial_degree.max()),
    )

    if condition == "dynamic_shuffle":
        permutation = _dynamic_permutation(
            battery_id,
            n_nodes,
            shuffle_id,
        )

        records = [
            {
                "fixation_id": rec["fixation_id"],
                "u": int(permutation[rec["u"]]),
                "v": int(permutation[rec["v"]]),
                "edge_type": rec["edge_type"],
            }
            for rec in true_records
        ]

        # Part-completion events are transformed by the same permutation.
        event_node_map = permutation

    else:
        permutation = np.arange(
            n_nodes,
            dtype=np.int64,
        )

        records = [
            dict(rec)
            for rec in true_records
        ]

        event_node_map = permutation

    # Static-real template.
    static_edge_index, static_edge_type, static_degree = (
        _state_edges_and_degree(
            n_nodes=n_nodes,
            records=records,
            removed_fixations=set(),
            active_nodes=np.ones(
                n_nodes,
                dtype=bool,
            ),
            initial_max_degree=initial_max_degree,
        )
    )

    removed_fixations = set()
    active_nodes = np.ones(
        n_nodes,
        dtype=bool,
    )
    part_completed = np.zeros(
        n_nodes,
        dtype=bool,
    )

    state_edge_indices = []
    state_edge_types = []
    state_degrees = []
    state_active_fraction = []
    state_edge_fraction = []

    initial_undirected_edges = len(
        true_records
    )

    seq = [
        int(x)
        for x in chains[
            battery_id
        ]
    ]

    for t in range(
        n_states
    ):
        # State is POST-current-operation: first apply operation t.
        op_id = seq[t]
        r = ops_by_id.loc[
            op_id
        ]

        if condition != "static_real":
            if pd.notna(
                r.fixation_id
            ):
                removed_fixations.add(
                    int(
                        r.fixation_id
                    )
                )
            else:
                semantic_node = (
                    part_id_to_local[
                        int(r.part_id)
                    ]
                )

                structural_node = int(
                    event_node_map[
                        semantic_node
                    ]
                )

                part_completed[
                    structural_node
                ] = True

            # Safe causal deactivation: part nodes disappear only once their
            # own part operation has happened AND no modeled fixation remains.
            active_nodes = _deactivate_completed_parts(
                records=records,
                removed_fixations=removed_fixations,
                active_nodes=active_nodes,
                part_completed=part_completed,
            )

        if condition == "static_real":
            edge_index = (
                static_edge_index
            )
            edge_type = (
                static_edge_type
            )
            degree_norm = (
                static_degree
            )

            active_fraction = 1.0
            edge_fraction = 1.0

        elif condition == "dynamic_noedge":
            edge_index = np.empty(
                (2, 0),
                dtype=np.int64,
            )
            edge_type = np.empty(
                (0,),
                dtype=np.int64,
            )
            degree_norm = np.zeros(
                n_nodes,
                dtype=np.float32,
            )

            active_fraction = float(
                active_nodes.mean()
            )
            edge_fraction = 0.0

        elif condition == "dynamic_complete":
            (
                edge_index,
                edge_type,
                degree_norm,
            ) = _complete_state_edges_and_degree(
                active_nodes
            )

            active_fraction = float(
                active_nodes.mean()
            )

            max_directed = (
                n_nodes
                * max(
                    0,
                    n_nodes - 1,
                )
            )

            edge_fraction = (
                edge_index.shape[1]
                / max(
                    1,
                    max_directed,
                )
            )

        else:
            (
                edge_index,
                edge_type,
                degree_norm,
            ) = _state_edges_and_degree(
                n_nodes=n_nodes,
                records=records,
                removed_fixations=removed_fixations,
                active_nodes=active_nodes,
                initial_max_degree=initial_max_degree,
            )

            active_fraction = float(
                active_nodes.mean()
            )

            edge_fraction = (
                (edge_index.shape[1] / 2.0)
                / max(
                    1,
                    initial_undirected_edges,
                )
            )

        state_edge_indices.append(
            edge_index
        )
        state_edge_types.append(
            edge_type
        )
        state_degrees.append(
            degree_norm
        )
        state_active_fraction.append(
            active_fraction
        )
        state_edge_fraction.append(
            edge_fraction
        )

    # Build one block-diagonal graph containing every ranking state.
    block_src = []
    block_dst = []
    block_type = []

    for state_idx, (
        edge_index,
        edge_type,
    ) in enumerate(
        zip(
            state_edge_indices,
            state_edge_types,
        )
    ):
        offset = (
            state_idx
            * n_nodes
        )

        if edge_index.shape[1]:
            block_src.append(
                edge_index[0]
                + offset
            )
            block_dst.append(
                edge_index[1]
                + offset
            )
            block_type.append(
                edge_type
            )

    if block_src:
        block_edge_index = np.asarray(
            [
                np.concatenate(
                    block_src
                ),
                np.concatenate(
                    block_dst
                ),
            ],
            dtype=np.int64,
        )

        block_edge_type = np.concatenate(
            block_type
        ).astype(
            np.int64
        )
    else:
        block_edge_index = np.empty(
            (2, 0),
            dtype=np.int64,
        )

        block_edge_type = np.empty(
            (0,),
            dtype=np.int64,
        )

    block_pack = {
        "node_part_class": np.tile(
            base_pack[
                "node_part_class"
            ],
            n_states,
        ).astype(np.int64),
        "node_degree": np.concatenate(
            state_degrees
        ).astype(np.float32),
        "edge_index": block_edge_index,
        "edge_type": block_edge_type,
    }

    return {
        "base_pack": base_pack,
        "state_graph_pack": block_pack,
        "n_nodes": int(
            n_nodes
        ),
        "n_states": int(
            n_states
        ),
        "state_active_fraction": np.asarray(
            state_active_fraction,
            dtype=np.float32,
        ),
        "state_edge_fraction": np.asarray(
            state_edge_fraction,
            dtype=np.float32,
        ),
        "condition": condition,
        "shuffle_id": int(
            shuffle_id
        ),
        "permutation": permutation.astype(
            np.int64
        ),
    }


## 14. Validate the sequence-induced state model

The original immediate-deactivation rule was too aggressive for the dataset: in the Fiat 500e sequence, the BMS `lift_off` operation is followed by two later `unplug` operations whose fixation records are still incident to that BMS. The table below explicitly detects such ordering cases.

The revised causal rule defers structural node deactivation until the part operation has occurred **and** no remaining modeled fixation is incident to that node. We then verify that every recorded next fixation is still represented and every recorded next part is still structurally active at the moment it is ranked.


In [ ]:
# 14. Dynamic-state validity diagnostics

def raw_part_before_later_fixation_cases(
    battery_id,
):
    psub = parts[
        parts["battery_id"]
        == battery_id
    ].copy()

    seq = [
        int(x)
        for x in chains[
            battery_id
        ]
    ]

    fixation_positions = {}

    for pos, op_id in enumerate(
        seq
    ):
        r = ops_by_id.loc[
            op_id
        ]

        if pd.notna(
            r.fixation_id
        ):
            fixation_positions[
                int(
                    r.fixation_id
                )
            ] = pos

    rows = []

    for pos, op_id in enumerate(
        seq
    ):
        r = ops_by_id.loc[
            op_id
        ]

        if pd.notna(
            r.fixation_id
        ):
            continue

        pid = int(
            r.part_id
        )

        later = []

        for fixation_id, fixation_pos in fixation_positions.items():
            if fixation_pos <= pos:
                continue

            fi = fix_info_raw[
                fixation_id
            ]

            if pid in {
                int(
                    fi[
                        "fromPartID"
                    ]
                ),
                int(
                    fi[
                        "toPartID"
                    ]
                ),
            }:
                later.append(
                    (
                        fixation_pos,
                        int(
                            seq[
                                fixation_pos
                            ]
                        ),
                        fixation_id,
                        str(
                            fi[
                                "class"
                            ]
                        ),
                    )
                )

        if later:
            rows.append({
                "battery_id": int(
                    battery_id
                ),
                "battery": battery_names[
                    battery_id
                ],
                "part_operation_position": int(
                    pos
                ),
                "part_operation_id": int(
                    op_id
                ),
                "part_id": int(
                    pid
                ),
                "part_class": str(
                    part_class_raw[
                        pid
                    ]
                ),
                "later_incident_fixation_count": int(
                    len(
                        later
                    )
                ),
                "later_incident_fixation_positions": ",".join(
                    str(
                        x[
                            0
                        ]
                    )
                    for x in later
                ),
                "later_incident_fixation_ids": ",".join(
                    str(
                        x[
                            2
                        ]
                    )
                    for x in later
                ),
            })

    return rows


ordering_cases = []

for b in sorted(
    chains
):
    ordering_cases.extend(
        raw_part_before_later_fixation_cases(
            b
        )
    )

ordering_cases = pd.DataFrame(
    ordering_cases
)

print(
    "Part operations followed by later incident fixation operations:",
    len(
        ordering_cases
    ),
)

if len(
    ordering_cases
):
    display(
        ordering_cases
    )


def e6_real_state_validity(
    battery_id,
):
    # Use a fold-valid vocabulary merely to build typed records.
    train_batteries = [
        b
        for b in sorted(
            chains
        )
        if b != battery_id
    ]

    vocab = build_fold_vocab(
        train_batteries
    )

    (
        psub,
        part_id_to_local,
        records,
    ) = _battery_structure_records(
        battery_id,
        vocab,
    )

    n_nodes = len(
        psub
    )

    active_nodes = np.ones(
        n_nodes,
        dtype=bool,
    )

    part_completed = np.zeros(
        n_nodes,
        dtype=bool,
    )

    removed_fixations = set()

    seq = [
        int(x)
        for x in chains[
            battery_id
        ]
    ]

    next_fix_valid = []
    next_part_valid = []

    for t in range(
        len(
            seq
        )
        - 1
    ):
        current = (
            ops_by_id.loc[
                seq[
                    t
                ]
            ]
        )

        if pd.notna(
            current.fixation_id
        ):
            removed_fixations.add(
                int(
                    current.fixation_id
                )
            )
        else:
            node = (
                part_id_to_local[
                    int(
                        current.part_id
                    )
                ]
            )

            part_completed[
                node
            ] = True

        active_nodes = (
            _deactivate_completed_parts(
                records=records,
                removed_fixations=removed_fixations,
                active_nodes=active_nodes,
                part_completed=part_completed,
            )
        )

        nxt = ops_by_id.loc[
            seq[
                t + 1
            ]
        ]

        if pd.notna(
            nxt.fixation_id
        ):
            fi_id = int(
                nxt.fixation_id
            )

            fi = fix_info_raw[
                fi_id
            ]

            u = part_id_to_local[
                int(
                    fi[
                        "fromPartID"
                    ]
                )
            ]

            v = part_id_to_local[
                int(
                    fi[
                        "toPartID"
                    ]
                )
            ]

            next_fix_valid.append(
                (
                    fi_id
                    not in removed_fixations
                )
                and bool(
                    active_nodes[
                        u
                    ]
                )
                and bool(
                    active_nodes[
                        v
                    ]
                )
            )

        else:
            node = (
                part_id_to_local[
                    int(
                        nxt.part_id
                    )
                ]
            )

            next_part_valid.append(
                bool(
                    active_nodes[
                        node
                    ]
                )
            )

    return {
        "battery_id": int(
            battery_id
        ),
        "battery": battery_names[
            battery_id
        ],
        "next_fixation_valid_rate": (
            float(
                np.mean(
                    next_fix_valid
                )
            )
            if next_fix_valid
            else np.nan
        ),
        "next_part_active_rate": (
            float(
                np.mean(
                    next_part_valid
                )
            )
            if next_part_valid
            else np.nan
        ),
        "n_next_fixation_checks": int(
            len(
                next_fix_valid
            )
        ),
        "n_next_part_checks": int(
            len(
                next_part_valid
            )
        ),
    }


validity_rows = [
    e6_real_state_validity(
        b
    )
    for b in sorted(
        chains
    )
]

state_validity = pd.DataFrame(
    validity_rows
)

display(
    state_validity
)

assert np.nanmin(
    state_validity[
        "next_fixation_valid_rate"
    ]
) == 1.0, (
    "At least one next fixation is absent even under deferred deactivation."
)

assert np.nanmin(
    state_validity[
        "next_part_active_rate"
    ]
) == 1.0, (
    "At least one next part is already inactive under deferred deactivation."
)

# Graph-evolution diagnostics.
progress_rows = []

for b in sorted(
    chains
):
    train_b = [
        x
        for x in sorted(
            chains
        )
        if x != b
    ]

    vocab_b = build_fold_vocab(
        train_b
    )

    pack = build_e6_state_pack(
        b,
        vocab_b,
        condition="dynamic_real",
    )

    n_states = pack[
        "n_states"
    ]

    for frac in [
        0.0,
        0.25,
        0.50,
        0.75,
        1.0,
    ]:
        idx = min(
            n_states - 1,
            int(
                round(
                    frac
                    * (
                        n_states
                        - 1
                    )
                )
            ),
        )

        progress_rows.append({
            "battery_id": int(
                b
            ),
            "battery": battery_names[
                b
            ],
            "progress": float(
                frac
            ),
            "active_node_fraction": float(
                pack[
                    "state_active_fraction"
                ][
                    idx
                ]
            ),
            "remaining_fixation_edge_fraction": float(
                pack[
                    "state_edge_fraction"
                ][
                    idx
                ]
            ),
        })

state_progress_diagnostics = pd.DataFrame(
    progress_rows
)

display(
    state_progress_diagnostics.groupby(
        "progress",
        as_index=False,
    ).agg(
        active_node_fraction=(
            "active_node_fraction",
            "mean",
        ),
        remaining_fixation_edge_fraction=(
            "remaining_fixation_edge_fraction",
            "mean",
        ),
    )
)

print(
    "Deferred-deactivation validity checks passed for all 10 batteries."
)


## 15. State-specific operation representations

The model parameters are exactly those of the canonical `GNNOperationRanker`.

The only difference is that the graph encoder receives the block-diagonal collection of state graphs. For an operation scored at state `t`, its structural representation is gathered only from block `t`. The operation-local MLP representation remains unchanged.


In [ ]:
# 15. Dynamic representation helpers

def build_e6_model(
    vocab
):
    return GNNOperationRanker(
        n_skill=len(
            vocab["skill"]
        ),
        n_fix=len(
            vocab["fix"]
        ),
        n_part=len(
            vocab["part"]
        ),
    ).to(DEVICE)


def compose_state_operation_repr(
    model,
    base_repr,
    node_h,
    base_pack,
    n_nodes,
    state_indices,
    op_indices,
):
    state_indices = torch.as_tensor(
        state_indices,
        dtype=torch.long,
        device=DEVICE,
    )

    op_indices = torch.as_tensor(
        op_indices,
        dtype=torch.long,
        device=DEVICE,
    )

    op_nodes_all = torch.as_tensor(
        base_pack[
            "op_graph_nodes"
        ],
        dtype=torch.long,
        device=DEVICE,
    )

    op_nodes = op_nodes_all[
        op_indices
    ]

    offsets = (
        state_indices
        * int(
            n_nodes
        )
    )

    n_items = len(
        op_indices
    )

    a = torch.zeros(
        (
            n_items,
            node_h.shape[1],
        ),
        dtype=node_h.dtype,
        device=DEVICE,
    )

    b = torch.zeros_like(
        a
    )

    fixation_mask = (
        op_nodes[:, 0]
        >= 0
    )

    if fixation_mask.any():
        a[
            fixation_mask
        ] = node_h[
            offsets[
                fixation_mask
            ]
            + op_nodes[
                fixation_mask,
                0,
            ]
        ]

        b[
            fixation_mask
        ] = node_h[
            offsets[
                fixation_mask
            ]
            + op_nodes[
                fixation_mask,
                1,
            ]
        ]

    part_mask = (
        op_nodes[:, 2]
        >= 0
    )

    if part_mask.any():
        a[
            part_mask
        ] = node_h[
            offsets[
                part_mask
            ]
            + op_nodes[
                part_mask,
                2,
            ]
        ]

    structural = (
        model.structural_projection(
            torch.cat(
                [
                    a,
                    b,
                    torch.abs(
                        a - b
                    ),
                ],
                dim=-1,
            )
        )
    )

    return model.combine(
        torch.cat(
            [
                base_repr[
                    op_indices
                ],
                structural,
            ],
            dim=-1,
        )
    )


## 16. Dynamic training and evaluation

The canonical pairwise objective is unchanged. Each training state uses its post-current graph, the recorded next operation as positive, and the same eight sampled future negatives.

The negative-sampling RNG and product order use the same fixed seed convention as the primary study.


In [ ]:
# 16. One E6 condition: train and evaluate

def train_one_e6_condition(
    train_batteries,
    test_battery,
    seed,
    condition,
    shuffle_id=0,
):
    set_seed(
        seed
    )

    vocab = build_fold_vocab(
        train_batteries
    )

    packs = {
        b: build_e6_state_pack(
            b,
            vocab,
            condition=condition,
            shuffle_id=shuffle_id,
        )
        for b in (
            train_batteries
            + [test_battery]
        )
    }

    model = build_e6_model(
        vocab
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    rng = np.random.default_rng(
        seed
    )

    for epoch in range(
        EPOCHS
    ):
        order = list(
            train_batteries
        )

        rng.shuffle(
            order
        )

        model.train()

        for b in order:
            item = packs[
                b
            ]

            base_pack = item[
                "base_pack"
            ]

            n_ops = base_pack[
                "n_ops"
            ]

            (
                cur,
                pos,
                neg,
                prog,
            ) = make_training_pairs(
                n_ops,
                NEGATIVES_PER_STATE,
                rng,
            )

            # Training states are t = 0,...,n_ops-3.
            state_idx = cur.copy()

            base_repr = (
                model.base_operation_encoding(
                    base_pack
                )
            )

            node_h = (
                model.graph_encoder(
                    item[
                        "state_graph_pack"
                    ]
                )
            )

            cur_repr = (
                compose_state_operation_repr(
                    model=model,
                    base_repr=base_repr,
                    node_h=node_h,
                    base_pack=base_pack,
                    n_nodes=item[
                        "n_nodes"
                    ],
                    state_indices=state_idx,
                    op_indices=cur,
                )
            )

            pos_repr = (
                compose_state_operation_repr(
                    model=model,
                    base_repr=base_repr,
                    node_h=node_h,
                    base_pack=base_pack,
                    n_nodes=item[
                        "n_nodes"
                    ],
                    state_indices=state_idx,
                    op_indices=pos,
                )
            )

            prog_t = torch.as_tensor(
                prog,
                dtype=torch.float32,
                device=DEVICE,
            ).unsqueeze(
                -1
            )

            pos_score = (
                model.score_pairs(
                    cur_repr,
                    pos_repr,
                    prog_t,
                )
            )

            k = neg.shape[1]

            neg_flat = (
                neg.reshape(
                    -1
                )
            )

            neg_state = np.repeat(
                state_idx,
                k,
            )

            neg_repr = (
                compose_state_operation_repr(
                    model=model,
                    base_repr=base_repr,
                    node_h=node_h,
                    base_pack=base_pack,
                    n_nodes=item[
                        "n_nodes"
                    ],
                    state_indices=neg_state,
                    op_indices=neg_flat,
                )
            )

            cur_rep = (
                cur_repr[
                    :,
                    None,
                    :,
                ]
                .expand(
                    -1,
                    k,
                    -1,
                )
                .reshape(
                    -1,
                    cur_repr.shape[
                        1
                    ],
                )
            )

            prog_rep = (
                prog_t[
                    :,
                    None,
                    :,
                ]
                .expand(
                    -1,
                    k,
                    -1,
                )
                .reshape(
                    -1,
                    1,
                )
            )

            neg_score = (
                model.score_pairs(
                    cur_rep,
                    neg_repr,
                    prog_rep,
                )
                .reshape(
                    len(cur),
                    k,
                )
            )

            loss = F.softplus(
                -(
                    pos_score[
                        :,
                        None,
                    ]
                    - neg_score
                )
            ).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return (
        model,
        packs[
            test_battery
        ],
        vocab,
    )


def evaluate_e6_model(
    model,
    item,
    train_batteries,
):
    model.eval()

    base_pack = item[
        "base_pack"
    ]

    seen_signatures = (
        training_signature_set(
            train_batteries
        )
    )

    rows = []

    with torch.no_grad():
        base_repr = (
            model.base_operation_encoding(
                base_pack
            )
        )

        node_h = (
            model.graph_encoder(
                item[
                    "state_graph_pack"
                ]
            )
        )

        n_ops = base_pack[
            "n_ops"
        ]

        for t in range(
            n_ops - 1
        ):
            candidate_indices = np.arange(
                t + 1,
                n_ops,
                dtype=np.int64,
            )

            state_current = np.asarray(
                [t],
                dtype=np.int64,
            )

            current_repr = (
                compose_state_operation_repr(
                    model=model,
                    base_repr=base_repr,
                    node_h=node_h,
                    base_pack=base_pack,
                    n_nodes=item[
                        "n_nodes"
                    ],
                    state_indices=state_current,
                    op_indices=np.asarray(
                        [t],
                        dtype=np.int64,
                    ),
                )
            )

            cand_states = np.full(
                len(
                    candidate_indices
                ),
                t,
                dtype=np.int64,
            )

            candidate_repr = (
                compose_state_operation_repr(
                    model=model,
                    base_repr=base_repr,
                    node_h=node_h,
                    base_pack=base_pack,
                    n_nodes=item[
                        "n_nodes"
                    ],
                    state_indices=cand_states,
                    op_indices=candidate_indices,
                )
            )

            current_rep = (
                current_repr.expand(
                    len(
                        candidate_indices
                    ),
                    -1,
                )
            )

            progress = torch.full(
                (
                    len(
                        candidate_indices
                    ),
                    1,
                ),
                t
                / max(
                    1,
                    n_ops - 1,
                ),
                dtype=torch.float32,
                device=DEVICE,
            )

            scores = (
                model.score_pairs(
                    current_rep,
                    candidate_repr,
                    progress,
                )
                .cpu()
                .numpy()
            )

            h1, h3, mrr = (
                expected_tie_metrics(
                    scores,
                    target_position=0,
                )
            )

            target_signature = (
                base_pack[
                    "full_signatures"
                ][
                    t + 1
                ]
            )

            rows.append({
                "position": int(
                    t
                ),
                "hits1": float(
                    h1
                ),
                "hits3": float(
                    h3
                ),
                "mrr": float(
                    mrr
                ),
                "signature_seen_in_training": bool(
                    target_signature
                    in seen_signatures
                ),
                "active_node_fraction": float(
                    item[
                        "state_active_fraction"
                    ][t]
                ),
                "edge_fraction": float(
                    item[
                        "state_edge_fraction"
                    ][t]
                ),
            })

    detail = pd.DataFrame(
        rows
    )

    metrics = {
        "hits1": float(
            detail[
                "hits1"
            ].mean()
        ),
        "hits3": float(
            detail[
                "hits3"
            ].mean()
        ),
        "mrr": float(
            detail[
                "mrr"
            ].mean()
        ),
        "seen_signature_rate": float(
            detail[
                "signature_seen_in_training"
            ].mean()
        ),
    }

    return (
        metrics,
        detail,
    )


## 17. Define the manufacturer-blocked folds

Manufacturer groups are defined explicitly from the battery names in the dataset.
For each affected held-out product, the training set is exactly the set of all
products **outside that manufacturer's group**. Assertions below prevent accidental
same-manufacturer leakage.


In [ ]:
# 17. Manufacturer-blocked fold design

battery_ids = sorted(chains)
name_to_id = {
    str(name): int(bid)
    for bid, name in battery_names.items()
}

required_names = {
    "BMW": [
        "BMW 530e",
        "BMW 330e",
        "BMW i3",
    ],
    "Chevrolet": [
        "Chevrolet Spark",
        "Chevrolet Volt",
    ],
}

for manufacturer, names in required_names.items():
    missing = [
        name
        for name in names
        if name not in name_to_id
    ]
    assert not missing, (
        f"Missing expected {manufacturer} products: {missing}"
    )

manufacturer_groups = {
    manufacturer: [
        name_to_id[name]
        for name in names
    ]
    for manufacturer, names in required_names.items()
}

affected_products = [
    bid
    for ids in manufacturer_groups.values()
    for bid in ids
]

assert len(affected_products) == 5
assert len(set(affected_products)) == 5

product_manufacturer = {
    bid: manufacturer
    for manufacturer, ids in manufacturer_groups.items()
    for bid in ids
}

e8_train_sets = {}
fold_rows = []

for test_battery in affected_products:
    manufacturer = product_manufacturer[test_battery]
    excluded_group = set(
        manufacturer_groups[manufacturer]
    )

    # Exclude the held-out product and every same-manufacturer sibling.
    train_batteries = [
        b
        for b in battery_ids
        if b not in excluded_group
    ]

    # Strong leakage assertions.
    assert test_battery not in train_batteries
    assert excluded_group.isdisjoint(train_batteries)
    assert set(train_batteries) == (
        set(battery_ids) - excluded_group
    )

    expected_n_train = (
        7
        if manufacturer == "BMW"
        else 8
    )
    assert len(train_batteries) == expected_n_train

    e8_train_sets[test_battery] = train_batteries

    fold_rows.append({
        "battery_id": int(test_battery),
        "battery": battery_names[test_battery],
        "manufacturer_group": manufacturer,
        "n_train_products": int(len(train_batteries)),
        "excluded_same_manufacturer_products": "; ".join(
            battery_names[b]
            for b in sorted(excluded_group - {test_battery})
        ),
        "training_products": "; ".join(
            battery_names[b]
            for b in train_batteries
        ),
        "training_ids": ";".join(
            str(int(b))
            for b in train_batteries
        ),
    })

fold_design = pd.DataFrame(fold_rows).sort_values(
    ["manufacturer_group", "battery"]
).reset_index(drop=True)

display(fold_design)


## 18. Validate the E8 DynamicReal / DynamicShuffled controls

Before training, this cell checks every affected product and all five shuffle
realizations. It verifies:

- operation-local categorical and numeric inputs are identical;
- the post-current active-node and remaining-edge fractions are identical;
- the dynamic shuffled block graph is exactly the real block graph under the fixed
  node permutation within every state;
- dynamic degree vectors transform by the same permutation;
- edge-type sequences are unchanged.

These assertions are stronger than checking only aggregate edge counts.


In [ ]:
# 18. E8 control-construction validity checks

control_rows = []

for test_battery in affected_products:
    train_batteries = e8_train_sets[test_battery]
    vocab = build_fold_vocab(train_batteries)

    real_pack = build_e6_state_pack(
        test_battery,
        vocab,
        condition="dynamic_real",
        shuffle_id=0,
    )

    n_nodes = int(real_pack["n_nodes"])
    n_states = int(real_pack["n_states"])

    real_graph = real_pack["state_graph_pack"]
    real_edge_index = real_graph["edge_index"]
    real_edge_type = real_graph["edge_type"]
    real_degree = real_graph["node_degree"].reshape(
        n_states,
        n_nodes,
    )

    for sid in SHUFFLE_IDS:
        shuffled_pack = build_e6_state_pack(
            test_battery,
            vocab,
            condition="dynamic_shuffle",
            shuffle_id=sid,
        )

        # Same operation-local features.
        assert np.array_equal(
            real_pack["base_pack"]["op_categorical"],
            shuffled_pack["base_pack"]["op_categorical"],
        )
        assert np.allclose(
            real_pack["base_pack"]["op_numeric"],
            shuffled_pack["base_pack"]["op_numeric"],
        )
        assert np.array_equal(
            real_pack["base_pack"]["op_graph_nodes"],
            shuffled_pack["base_pack"]["op_graph_nodes"],
        )

        # Same dynamic amount of remaining structure.
        assert np.allclose(
            real_pack["state_active_fraction"],
            shuffled_pack["state_active_fraction"],
        )
        assert np.allclose(
            real_pack["state_edge_fraction"],
            shuffled_pack["state_edge_fraction"],
        )

        permutation = shuffled_pack["permutation"]
        assert sorted(permutation.tolist()) == list(range(n_nodes))

        shuffled_graph = shuffled_pack["state_graph_pack"]
        shuffled_edge_index = shuffled_graph["edge_index"]
        shuffled_edge_type = shuffled_graph["edge_type"]

        # Exact block-graph isomorphism under the state-wise node permutation.
        mapped_edge_index = real_edge_index.copy()
        for row in (0, 1):
            global_nodes = real_edge_index[row]
            state_idx = global_nodes // n_nodes
            local_node = global_nodes % n_nodes
            mapped_edge_index[row] = (
                state_idx * n_nodes
                + permutation[local_node]
            )

        assert np.array_equal(
            mapped_edge_index,
            shuffled_edge_index,
        )
        assert np.array_equal(
            real_edge_type,
            shuffled_edge_type,
        )

        # Dynamic degree vector follows the same permutation in every state.
        shuffled_degree = shuffled_graph["node_degree"].reshape(
            n_states,
            n_nodes,
        )

        expected_shuffled_degree = np.empty_like(real_degree)
        expected_shuffled_degree[:, permutation] = real_degree

        assert np.allclose(
            expected_shuffled_degree,
            shuffled_degree,
        )

        control_rows.append({
            "battery_id": int(test_battery),
            "battery": battery_names[test_battery],
            "manufacturer_group": product_manufacturer[test_battery],
            "shuffle_id": int(sid),
            "operation_local_inputs_identical": True,
            "dynamic_fraction_sequence_identical": True,
            "exact_block_graph_isomorphism": True,
            "degree_permutation_valid": True,
            "edge_types_identical": True,
        })

control_validity = pd.DataFrame(control_rows)

assert len(control_validity) == 25
assert control_validity[
    [
        "operation_local_inputs_identical",
        "dynamic_fraction_sequence_identical",
        "exact_block_graph_isomorphism",
        "degree_permutation_valid",
        "edge_types_identical",
    ]
].all().all()

display(control_validity)
print("All E8 control-construction checks passed.")


## 19. Run the targeted E8 experiment

Total training burden is **90 neural fits**:

\[
5\ \text{held-out products}
\times 3\ \text{seeds}
\times (1\ \text{DynamicReal} + 5\ \text{DynamicShuffled})
= 90.
\]

A partial CSV is written after every fit so the run can be resumed safely.


In [ ]:
# 19. Full E8 manufacturer-blocked run

import itertools
import shutil

OUT = Path(
    "/content/battery_e8_manufacturer_exclusion"
)
OUT.mkdir(
    parents=True,
    exist_ok=True,
)

PARTIAL = (
    OUT
    / "e8_partial.csv"
)

E8_DESIGN_ID = "manufacturer_blocked_dynamic_real_vs_shuffle_v1"

conditions = [
    (
        "DynamicRealGNN",
        "dynamic_real",
        0,
    ),
] + [
    (
        "DynamicShuffledGNN",
        "dynamic_shuffle",
        sid,
    )
    for sid in SHUFFLE_IDS
]

required_partial_columns = {
    "design_id",
    "battery_id",
    "seed",
    "condition",
    "shuffle_id",
    "training_ids",
}

if PARTIAL.exists():
    results = pd.read_csv(PARTIAL)

    assert required_partial_columns.issubset(
        set(results.columns)
    ), (
        "Existing E8 partial file has an incompatible schema. "
        "Delete it or use a clean output directory."
    )

    assert set(results["design_id"].astype(str)) == {
        E8_DESIGN_ID
    }, (
        "Existing E8 partial file was created by a different design."
    )

    print(
        "Resuming from",
        len(results),
        "completed fits.",
    )
else:
    results = pd.DataFrame()


def e8_fit_done(
    df,
    test_battery,
    seed,
    condition,
    shuffle_id,
    training_ids,
):
    if len(df) == 0:
        return False

    mask = (
        (df["design_id"].astype(str) == E8_DESIGN_ID)
        & (df["battery_id"] == int(test_battery))
        & (df["seed"] == int(seed))
        & (df["condition"].astype(str) == str(condition))
        & (df["shuffle_id"] == int(shuffle_id))
        & (df["training_ids"].astype(str) == str(training_ids))
    )

    return bool(mask.any())


start_time = time.time()

for fold_idx, test_battery in enumerate(
    affected_products,
    start=1,
):
    train_batteries = e8_train_sets[
        test_battery
    ]

    manufacturer = product_manufacturer[
        test_battery
    ]

    training_ids = ";".join(
        str(int(b))
        for b in train_batteries
    )

    # Reassert the manufacturer block immediately before training.
    excluded_group = set(
        manufacturer_groups[manufacturer]
    )
    assert excluded_group.isdisjoint(
        train_batteries
    )

    print(
        f"\nE8 fold {fold_idx}/{len(affected_products)} "
        f"— held out: {battery_names[test_battery]} "
        f"({manufacturer}); "
        f"n_train={len(train_batteries)}"
    )

    for (
        label,
        condition,
        shuffle_id,
    ) in conditions:
        for seed in SEEDS:
            if e8_fit_done(
                results,
                test_battery,
                seed,
                condition,
                shuffle_id,
                training_ids,
            ):
                print(
                    f"  skip {condition} "
                    f"s={shuffle_id} "
                    f"seed={seed}"
                )
                continue

            (
                model,
                test_item,
                vocab,
            ) = train_one_e6_condition(
                train_batteries=train_batteries,
                test_battery=test_battery,
                seed=seed,
                condition=condition,
                shuffle_id=shuffle_id,
            )

            metrics, detail = evaluate_e6_model(
                model,
                test_item,
                train_batteries,
            )

            row = {
                "design_id": E8_DESIGN_ID,
                "battery_id": int(test_battery),
                "battery": battery_names[test_battery],
                "manufacturer_group": manufacturer,
                "n_train_products": int(len(train_batteries)),
                "training_ids": training_ids,
                "model": label,
                "condition": condition,
                "shuffle_id": int(shuffle_id),
                "seed": int(seed),
                "hits1": float(metrics["hits1"]),
                "hits3": float(metrics["hits3"]),
                "mrr": float(metrics["mrr"]),
                "seen_signature_rate": float(
                    metrics["seen_signature_rate"]
                ),
                "parameter_count": int(
                    count_trainable_parameters(model)
                ),
                "mean_active_node_fraction": float(
                    test_item[
                        "state_active_fraction"
                    ].mean()
                ),
                "mean_edge_fraction": float(
                    test_item[
                        "state_edge_fraction"
                    ].mean()
                ),
            }

            results = pd.concat(
                [
                    results,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            results.to_csv(
                PARTIAL,
                index=False,
            )

            print(
                f"  done {condition} "
                f"s={shuffle_id} "
                f"seed={seed}: "
                f"H1={metrics['hits1']:.3f}, "
                f"H3={metrics['hits3']:.3f}, "
                f"MRR={metrics['mrr']:.3f}"
            )

            del model

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

elapsed = (
    time.time()
    - start_time
) / 3600.0

assert len(results) == E8_EXPECTED_FITS, (
    f"E8 incomplete: {len(results)} fits present; "
    f"{E8_EXPECTED_FITS} expected."
)

# Every affected product must have exactly 18 fits:
# 3 seeds × (1 real + 5 shuffled).
per_product_fit_count = (
    results
    .groupby("battery_id")
    .size()
)
assert (per_product_fit_count == 18).all()

print(
    f"\nCompleted/resumed to {len(results)} fits "
    f"in {elapsed:.2f} hours."
)


## 20. Product-level aggregation

The five shuffled realizations are first averaged **within product and seed**.
Neural seeds are then averaged **within product**. This preserves the held-out battery
product as the independent unit and avoids treating seeds or shuffle realizations as
independent observations.


In [ ]:
# 20. Aggregate E8 within product

seed_rows = []

for (
    battery_id,
    battery,
    manufacturer_group,
    seed,
), grp in results.groupby(
    [
        "battery_id",
        "battery",
        "manufacturer_group",
        "seed",
    ]
):
    real = grp[
        grp["condition"]
        == "dynamic_real"
    ]

    assert len(real) == 1

    seed_rows.append({
        "battery_id": int(battery_id),
        "battery": battery,
        "manufacturer_group": manufacturer_group,
        "seed": int(seed),
        "condition_summary": "dynamic_real",
        "hits1": float(real["hits1"].iloc[0]),
        "hits3": float(real["hits3"].iloc[0]),
        "mrr": float(real["mrr"].iloc[0]),
        "realization_sd_hits1": 0.0,
    })

    shuffled = grp[
        grp["condition"]
        == "dynamic_shuffle"
    ]

    assert len(shuffled) == len(SHUFFLE_IDS)
    assert set(
        shuffled["shuffle_id"].astype(int)
    ) == set(SHUFFLE_IDS)

    seed_rows.append({
        "battery_id": int(battery_id),
        "battery": battery,
        "manufacturer_group": manufacturer_group,
        "seed": int(seed),
        "condition_summary": "dynamic_shuffle_mean",
        "hits1": float(shuffled["hits1"].mean()),
        "hits3": float(shuffled["hits3"].mean()),
        "mrr": float(shuffled["mrr"].mean()),
        "realization_sd_hits1": float(
            shuffled["hits1"].std()
        ),
    })

seed_summary = pd.DataFrame(seed_rows)

fold_mean = (
    seed_summary
    .groupby(
        [
            "battery_id",
            "battery",
            "manufacturer_group",
            "condition_summary",
        ],
        as_index=False,
    )
    .agg(
        hits1=("hits1", "mean"),
        hits3=("hits3", "mean"),
        mrr=("mrr", "mean"),
        realization_sd_hits1=(
            "realization_sd_hits1",
            "mean",
        ),
    )
)

aggregate = (
    fold_mean
    .groupby(
        "condition_summary",
        as_index=False,
    )
    .agg(
        hits1=("hits1", "mean"),
        hits3=("hits3", "mean"),
        mrr=("mrr", "mean"),
        mean_realization_sd_hits1=(
            "realization_sd_hits1",
            "mean",
        ),
    )
    .sort_values(
        "hits1",
        ascending=False,
    )
)

display(aggregate)
display(fold_mean)


## 21. E8 contrast and sensitivity summary

The designated E8 comparison is **DynamicReal − DynamicShuffled mean** on Hits@1.
Exact two-sided sign-flip p-values are computed at the five-product level for
transparency, but E8 is interpreted primarily through effect size and direction because
n=5 implies a minimum possible two-sided p-value of 0.0625.

Hits@3 and MRR are supporting descriptive endpoints. Bootstrap intervals are
descriptive.


In [ ]:
# 21. Product-level E8 contrasts

def exact_two_sided_sign_flip(values):
    values = np.asarray(values, dtype=float)

    observed = abs(values.mean())
    total = 2 ** len(values)
    extreme = 0

    for signs in itertools.product(
        (-1.0, 1.0),
        repeat=len(values),
    ):
        stat = abs(
            np.mean(
                values
                * np.asarray(signs)
            )
        )

        if stat >= observed - 1e-15:
            extreme += 1

    return extreme / total


def bootstrap_mean_ci(
    values,
    n_boot=20000,
    seed=20271010,
):
    values = np.asarray(values, dtype=float)

    rng = np.random.default_rng(seed)

    draws = rng.choice(
        values,
        size=(
            n_boot,
            len(values),
        ),
        replace=True,
    ).mean(axis=1)

    return (
        float(np.percentile(draws, 2.5)),
        float(np.percentile(draws, 97.5)),
    )


wide = fold_mean.pivot(
    index=[
        "battery_id",
        "battery",
        "manufacturer_group",
    ],
    columns="condition_summary",
    values=[
        "hits1",
        "hits3",
        "mrr",
    ],
)

assert len(wide) == 5

contrast_rows = []

for metric_idx, metric in enumerate(
    ["hits1", "hits3", "mrr"]
):
    d = (
        wide[
            (
                metric,
                "dynamic_real",
            )
        ]
        - wide[
            (
                metric,
                "dynamic_shuffle_mean",
            )
        ]
    ).to_numpy(float)

    ci = bootstrap_mean_ci(
        d,
        seed=20271010 + metric_idx,
    )

    contrast_rows.append({
        "metric": metric,
        "designated_primary_e8_endpoint": (
            metric == "hits1"
        ),
        "n_products": int(len(d)),
        "mean_difference_real_minus_shuffle": float(
            d.mean()
        ),
        "ci_low_descriptive": ci[0],
        "ci_high_descriptive": ci[1],
        "positive_products": int((d > 0).sum()),
        "negative_products": int((d < 0).sum()),
        "zero_products": int((d == 0).sum()),
        "p_exact_two_sided": float(
            exact_two_sided_sign_flip(d)
        ),
    })

contrast = pd.DataFrame(contrast_rows)

# Product-level table for the main sensitivity interpretation.
product_contrast = pd.DataFrame({
    "battery_id": wide.index.get_level_values(
        "battery_id"
    ).astype(int),
    "battery": wide.index.get_level_values(
        "battery"
    ),
    "manufacturer_group": wide.index.get_level_values(
        "manufacturer_group"
    ),
    "real_hits1": wide[
        (
            "hits1",
            "dynamic_real",
        )
    ].to_numpy(float),
    "shuffle_mean_hits1": wide[
        (
            "hits1",
            "dynamic_shuffle_mean",
        )
    ].to_numpy(float),
})

product_contrast[
    "real_minus_shuffle_hits1"
] = (
    product_contrast["real_hits1"]
    - product_contrast["shuffle_mean_hits1"]
)

product_contrast[
    "real_favored_hits1"
] = (
    product_contrast[
        "real_minus_shuffle_hits1"
    ]
    > 0
)

display(contrast)
display(product_contrast)

print(
    "Important: with n=5, the minimum possible "
    "two-sided exact sign-flip p-value is 0.0625."
)
print(
    "Interpret E8 as a robustness/sensitivity analysis, "
    "not a new confirmatory test."
)


## 22. Save E8 outputs

The ZIP contains the complete fit-level results, manufacturer-blocked fold design,
control-validity assertions, seed/product summaries, and the designated E8 contrast.


In [ ]:
# 22. Save E8 outputs

results.to_csv(
    OUT / "e8_all_fit_results.csv",
    index=False,
)

fold_design.to_csv(
    OUT / "e8_fold_design.csv",
    index=False,
)

control_validity.to_csv(
    OUT / "e8_control_validity.csv",
    index=False,
)

seed_summary.to_csv(
    OUT / "e8_seed_summary.csv",
    index=False,
)

fold_mean.to_csv(
    OUT / "e8_fold_mean.csv",
    index=False,
)

aggregate.to_csv(
    OUT / "e8_aggregate.csv",
    index=False,
)

contrast.to_csv(
    OUT / "e8_contrast.csv",
    index=False,
)

product_contrast.to_csv(
    OUT / "e8_product_contrast.csv",
    index=False,
)

metadata = {
    "experiment": (
        "E8 same-manufacturer exclusion sensitivity analysis"
    ),
    "basis": (
        "Frozen E6 v2 DynamicReal and DynamicShuffled implementation"
    ),
    "independent_unit": (
        "held-out battery product; five affected products"
    ),
    "manufacturer_groups": {
        manufacturer: [
            battery_names[b]
            for b in ids
        ]
        for manufacturer, ids in manufacturer_groups.items()
    },
    "conditions": [
        "dynamic_real",
        "dynamic_shuffle x5",
    ],
    "seeds": [
        int(x)
        for x in SEEDS
    ],
    "shuffle_ids": [
        int(x)
        for x in SHUFFLE_IDS
    ],
    "epochs": int(EPOCHS),
    "negative_samples_per_state": int(
        NEGATIVES_PER_STATE
    ),
    "learning_rate": float(LR),
    "weight_decay": float(WEIGHT_DECAY),
    "dropout": float(DROPOUT),
    "state_convention": (
        "post-current; operation t is applied before ranking "
        "future candidates with t+1 as reference positive"
    ),
    "dynamic_update": (
        "executed fixation removes its fixation edge; "
        "executed part marks removal-complete; node deactivation "
        "is deferred until no modeled unexecuted incident fixation remains"
    ),
    "primary_e8_contrast": (
        "DynamicReal minus five-shuffle mean on Hits@1"
    ),
    "inference_note": (
        "Sensitivity analysis only. n=5 affected products; "
        "minimum possible two-sided exact sign-flip p-value is 0.0625. "
        "Effect magnitude and directional consistency are emphasized. "
        "Bootstrap intervals are descriptive."
    ),
}

with open(
    OUT / "e8_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

LIGHT = Path(
    "/content/e8_manufacturer_exclusion_results"
)

if LIGHT.exists():
    shutil.rmtree(LIGHT)

LIGHT.mkdir(
    parents=True,
    exist_ok=True,
)

for fn in [
    "e8_all_fit_results.csv",
    "e8_fold_design.csv",
    "e8_control_validity.csv",
    "e8_seed_summary.csv",
    "e8_fold_mean.csv",
    "e8_aggregate.csv",
    "e8_contrast.csv",
    "e8_product_contrast.csv",
    "e8_metadata.json",
]:
    shutil.copy2(
        OUT / fn,
        LIGHT / fn,
    )

zip_path = shutil.make_archive(
    "/content/e8_manufacturer_exclusion_results",
    "zip",
    root_dir=LIGHT,
)

print("Saved:", zip_path)

for p in sorted(LIGHT.iterdir()):
    print(" -", p.name)
